<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_07_model_training/stage_07_01_naive_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage_07_01 - NAIVE  (Seq2Seq 29x8 → 29x1)

Esta notebook entrena y evalúa el modelo **NAIVE** para el problema seq2seq:

- **Entrada:** (29 x 8)
- **Salida:** (29 x 1) con $ \Delta pts_h\ $ por minuto (h = 60 o 90)

**Output:** métricas y predicciones out-of-sample guardadas como artefactos para el **Stage_08**.

## **1. Imports + paths**

In [77]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

In [78]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:

#VENTANAS 60MIN
IN_WINDOW_TRAIN_60_Z = Path(os.environ.get("IN_WINDOW_TRAIN_60_Z", "data/windows/scaled/windows_train_60_z.npz"))
IN_WINDOW_VALID_60_Z = Path(os.environ.get("IN_WINDOW_VALID_60_Z", "data/windows/scaled/windows_valid_60_z.npz"))
IN_WINDOW_TEST_60_Z = Path(os.environ.get("IN_WINDOW_TEST_60_Z", "data/windows/scaled/windows_test_60_z.npz"))
IN_SCALER_60 = Path(os.environ.get("IN_SCALER_60", "data/windows/scaled/scaler_60.pkl"))

#VENTANAS 90MIN
IN_WINDOW_TRAIN_90_Z = Path(os.environ.get("IN_WINDOW_TRAIN_90_Z", "data/windows/scaled/windows_train_90_z.npz"))
IN_WINDOW_VALID_90_Z = Path(os.environ.get("IN_WINDOW_VALID_90_Z", "data/windows/scaled/windows_valid_90_z.npz"))
IN_WINDOW_TEST_90_Z = Path(os.environ.get("IN_WINDOW_TEST_90_Z", "data/windows/scaled/windows_test_90_z.npz"))
IN_SCALER_90 = Path(os.environ.get("IN_SCALER_90", "data/windows/scaled/scaler_90.pkl"))

#ARTIFACTS

# Summary del stage_03a (donde está delta_target_p70 por horizonte).
IN_TARGET_INVESTIGATION_SUMMARY = Path(os.environ.get("IN_TARGET_INVESTIGATION_SUMMARY", "reports/stage_03a_target_investigation_summary.json"))

# Summary del stage_06 (donde está window_size y n_features por horizonte).
IN_WINDOWS_SCALING_SUMMARY =Path(os.environ.get("IN_WINDOWS_SCALING_SUMMARY", "reports/stage_06_window_scaling_seq2seq_summary.json"))

#OUT_MODEL_METRICS = Path(os.environ.get("OUT_MODEL_METRICS", f"reports/stage_07__model_metrics.json"))

In [79]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [80]:
#PARA LA NOTEBOOK
IN_WINDOW_TRAIN_60_Z = DRIVE_DIR / IN_WINDOW_TRAIN_60_Z
IN_WINDOW_VALID_60_Z = DRIVE_DIR / IN_WINDOW_VALID_60_Z
IN_WINDOW_TEST_60_Z = DRIVE_DIR / IN_WINDOW_TEST_60_Z
IN_SCALER_60 = DRIVE_DIR / IN_SCALER_60

IN_WINDOW_TRAIN_90_Z = DRIVE_DIR / IN_WINDOW_TRAIN_90_Z
IN_WINDOW_VALID_90_Z=DRIVE_DIR / IN_WINDOW_VALID_90_Z
IN_WINDOW_TEST_90_Z = DRIVE_DIR / IN_WINDOW_TEST_90_Z
IN_SCALER_90 = DRIVE_DIR / IN_SCALER_90

IN_WINDOWS_SCALING_SUMMARY = DRIVE_DIR / IN_WINDOWS_SCALING_SUMMARY
IN_TARGET_INVESTIGATION_SUMMARY = DRIVE_DIR / IN_TARGET_INVESTIGATION_SUMMARY

## **2. Reproducibilidad**

In [81]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **3. Configuración**

In [82]:
def _read_json(path: Path) -> Dict[str, Any]:
    """Lee un JSON y devuelve un dict Python (con validación básica de existencia)."""
    # Verifica que el archivo exista antes de abrirlo.
    if not path.exists():
        # Si no existe, corta la ejecución con un error claro.
        raise FileNotFoundError(f"No existe el JSON: {path}")
    # Abre el archivo en modo lectura, asegurando UTF-8.
    with path.open("r", encoding="utf-8") as f:
        # Parsea el contenido JSON y lo devuelve como dict.
        return json.load(f)

In [83]:
# Config final para entrenar (modelo, horizonte).
@dataclass(frozen=True)
class StageConfig:
    # Horizonte (60 o 90).
    horizon: int
    # Largo de ventana (timesteps) desde stage_06.
    seq_len: int
    # Cantidad de features desde stage_06 para ese horizonte.
    n_features: int
    # Nombres de features (orden exacto) para ese horizonte.
    feature_names: List[str]
    # Nombre del target para ese horizonte.
    target_name: List[str]
    # Umbral mínimo económico (DELTA_BASE).
    delta_base: float
    # Umbral de oportunidad (DELTA_OP) leído del stage_03a.
    delta_op: float

In [84]:
# Construye un StageConfig leyendo ambos reports.
def load_state_from_reports(
    horizon: int,
    #model_name: str,
    *,
    in_windows_scaling_summary: Path = IN_WINDOWS_SCALING_SUMMARY,
    in_target_investigation_summary: Path = IN_TARGET_INVESTIGATION_SUMMARY,
) -> StageConfig:
    # Valida horizonte permitido.
    if horizon not in (60, 90):
        raise ValueError("horizon debe ser 60 o 90")

    # Carga JSON del stage_06.
    w = _read_json(in_windows_scaling_summary)

    # Carga JSON del stage_03a.
    t = _read_json(in_target_investigation_summary)

    # Lee window_size global (SEQ_LEN).
    seq_len = int(w["details"]["config"]["window_size"])

    # Selecciona dataset del horizonte (ojo: "60" o "90" como string).
    ds = w["details"]["datasets"][str(horizon)]

    # Lee n_features del horizonte.
    n_features = int(ds["n_features"])

    # Lee feature_names del horizonte (aquí se refleja el 1 feature distinto).
    feature_names = list(ds["feature_names"])

    # Lee target del horizonte.
    target_name = ds["target"]

    # Valida consistencia.
    if len(feature_names) != n_features:
        raise ValueError("Inconsistencia entre n_features y feature_names")

    # Define la key de delta_base_med del stage_03a.
    target_base = f"h{horizon}_delta_base_med"
    delta_base = float(t["metrics"][target_base])

    # Define la key de delta_target_p70 del stage_03a.
    target_op = f"h{horizon}_delta_target_p70"
    # Lee DELTA_OP para ese horizonte.
    delta_op = float(t["metrics"][target_op])

    # Devuelve la config lista para entrenar.
    return StageConfig(
        horizon=horizon,
        seq_len=seq_len,
        n_features=n_features,
        feature_names=feature_names,
        target_name=target_name,
        delta_base=float(delta_base),
        delta_op=float(delta_op),
            )


In [85]:
states_h60 = load_state_from_reports(horizon=60)
states_h60

StageConfig(horizon=60, seq_len=29, n_features=7, feature_names=['open', 'high', 'low', 'close', 'ema_60', 'mom_10_struct', 'roc_60'], target_name='delta_pts_60', delta_base=52.12, delta_op=84.18)

In [86]:
states_h90 = load_state_from_reports(horizon=90)
states_h90

StageConfig(horizon=90, seq_len=29, n_features=7, feature_names=['open', 'high', 'low', 'close', 'ema_60', 'mom_10_struct', 'roc_30'], target_name='delta_pts_90', delta_base=60.75, delta_op=97.22)


## **4. Importar métricas comunes desde .py**

In [87]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2seq_metrics import compute_seq2seq_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [88]:
print(compute_seq2seq_metrics.__doc__)


    Calcula métricas simples y comparables para modelos seq2seq.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples, seq_len) o (n_samples, seq_len, 1)
    y_pred : np.ndarray
        Valores predichos con shape (n_samples, seq_len) o (n_samples, seq_len, 1)
    compute_r2 : bool
        Si True, calcula R² sobre la secuencia completa concatenada.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 (y_true_last==0 o y_pred_last==0)
        al calcular DA_last. Esto evita ambigüedad en la dirección.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales y diagnóstico por paso.
    


## **6. Carga de data windows**

In [89]:
# --------------------------------------------------
# Función común: carga .npz estándar (X, y)
# --------------------------------------------------
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.

    Espera claves:
    - 'X': array (n_samples, seq_len, n_features)
    - 'y' o 'Y': array (n_samples, seq_len) o (n_samples, seq_len, 1)
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Carga el NPZ (lectura).
    data = np.load(path)

    # Lee X (obligatoria).
    if "X" not in data:
        raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
    X = data["X"]

    # Lee y: soporta 'y' (convención usada) o 'Y' (por compatibilidad).
    if "y" in data:
        y = data["y"]
    elif "Y" in data:
        y = data["Y"]
    else:
        raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

    # Devuelve X e y.
    return X, y

In [90]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [91]:
# --------------------------------------------------
# Carga completa: train/valid/test + scaler por horizonte
# --------------------------------------------------
def load_windows_and_scaler_for_horizon(horizon: int) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler para un horizonte dado (60 o 90).

    Retorna un dict:
    {
      "horizon": 60,
      "paths": {...},
      "scaler": <StandardScaler>,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }
    """
    # Valida horizonte permitido.
    if horizon not in (60, 90):
        raise ValueError("horizon debe ser 60 o 90")

    # Selecciona paths según horizonte.
    if horizon == 60:
        train_path = IN_WINDOW_TRAIN_60_Z
        valid_path = IN_WINDOW_VALID_60_Z
        test_path  = IN_WINDOW_TEST_60_Z
        scaler_path = IN_SCALER_60
    else:
        train_path = IN_WINDOW_TRAIN_90_Z
        valid_path = IN_WINDOW_VALID_90_Z
        test_path  = IN_WINDOW_TEST_90_Z
        scaler_path = IN_SCALER_90

    # Carga ventanas.
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test,  y_test  = load_npz_windows(test_path)

    # Carga scaler.
    scaler = load_scaler(scaler_path)

    # Retorna todo empaquetado.
    return {
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }

In [92]:
# --------------------------------------------------
# Carga efectiva: H60 y H90 (dos datasets distintos)
# --------------------------------------------------

# Carga todo para 60 min.
bundle_60 = load_windows_and_scaler_for_horizon(60)

# Carga todo para 90 min.
bundle_90 = load_windows_and_scaler_for_horizon(90)


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [93]:
# --------------------------------------------------
# Verificación rápida
# --------------------------------------------------

# Shapes H60.
print("H60 Train:", bundle_60["train"]["X"].shape, bundle_60["train"]["y"].shape)
print("H60 Valid:", bundle_60["valid"]["X"].shape, bundle_60["valid"]["y"].shape)
print("H60 Test :", bundle_60["test"]["X"].shape,  bundle_60["test"]["y"].shape)

# Shapes H90.
print("H90 Train:", bundle_90["train"]["X"].shape, bundle_90["train"]["y"].shape)
print("H90 Valid:", bundle_90["valid"]["X"].shape, bundle_90["valid"]["y"].shape)
print("H90 Test :", bundle_90["test"]["X"].shape,  bundle_90["test"]["y"].shape)

# Información útil (scaler).
print("Scaler H60:", type(bundle_60["scaler"]).__name__)
print("Scaler H90:", type(bundle_90["scaler"]).__name__)

H60 Train: (912, 29, 7) (912, 29)
H60 Valid: (195, 29, 7) (195, 29)
H60 Test : (196, 29, 7) (196, 29)
H90 Train: (912, 29, 7) (912, 29)
H90 Valid: (195, 29, 7) (195, 29)
H90 Test : (196, 29, 7) (196, 29)
Scaler H60: StandardScaler
Scaler H90: StandardScaler


NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **7. Sanity Check**

In [94]:
def sanity_check(
    X: np.ndarray,                 # Tensores de entrada: (n_samples, seq_len, n_features)
    y: np.ndarray,                 # Targets: (n_samples, seq_len) o (n_samples, seq_len, 1)
    name: str,                     # Nombre lógico del split (ej: "train_h60", "valid_h90")
    *,
    expected_seq_len: int,         # Largo de secuencia esperado (ej: 29)
    expected_n_features: int,      # Número de features esperado (ej: 8)
) -> None:
    # --------------------------------------------------
    # Chequeos de dimensionalidad
    # --------------------------------------------------

    # X debe ser estrictamente 3D: (muestras, tiempo, features)
    assert X.ndim == 3, (
        f"{name}: X debe ser 3D (n, seq, feat)"
    )

    # y puede ser 2D (n, seq) o 3D (n, seq, 1)
    assert y.ndim in (2, 3), (
        f"{name}: y debe ser 2D o 3D (n, seq) o (n, seq, 1)"
    )

    # --------------------------------------------------
    # Chequeos de consistencia temporal y estructural
    # --------------------------------------------------

    # Verifica que el largo temporal de X coincida con el esperado
    assert X.shape[1] == expected_seq_len, (
        f"{name}: seq_len inesperado en X: {X.shape[1]} != {expected_seq_len}"
    )

    # Verifica que la cantidad de features en X sea la esperada
    assert X.shape[2] == expected_n_features, (
        f"{name}: n_features inesperado en X: {X.shape[2]} != {expected_n_features}"
    )

    # Verifica que y tenga el mismo largo temporal que X
    assert y.shape[1] == expected_seq_len, (
        f"{name}: seq_len inesperado en y: {y.shape[1]} != {expected_seq_len}"
    )

    # --------------------------------------------------
    # Chequeos numéricos (sanidad de valores)
    # --------------------------------------------------

    # Asegura que X no contenga NaN ni infinitos
    assert np.isfinite(X).all(), (
        f"{name}: X contiene NaN/inf"
    )

    # Asegura que y no contenga NaN ni infinitos
    assert np.isfinite(y).all(), (
        f"{name}: y contiene NaN/inf"
    )

In [95]:
from __future__ import annotations

from typing import Any, Dict
import numpy as np


def run_sanity_checks_for_bundle(bundle: Dict[str, Any], *, tag: str) -> None:
    """
    Ejecuta sanity_check para train/valid/test usando únicamente variables locales.

    Espera un bundle con estructura:
    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
      ...
    }
    """
    # Extrae arrays localmente (no crea globals)
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]

    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]

    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    # Toma expected_* desde TRAIN (consistencia)
    expected_seq_len = int(X_tr.shape[1])
    expected_n_features = int(X_tr.shape[2])

    # Ejecuta sanity checks por split
    sanity_check(X_tr, y_tr, f"train_{tag}", expected_seq_len=expected_seq_len, expected_n_features=expected_n_features)
    sanity_check(X_va, y_va, f"valid_{tag}", expected_seq_len=expected_seq_len, expected_n_features=expected_n_features)
    sanity_check(X_te, y_te, f"test_{tag}",  expected_seq_len=expected_seq_len, expected_n_features=expected_n_features)

    # Mensaje de OK por bundle/horizonte
    h = bundle.get("horizon", "NA")
    print(f"OK {tag} (h={h}) | seq_len={expected_seq_len} | n_features={expected_n_features}")

    # Opcional: borra referencias locales explícitamente (no es estrictamente necesario)
    del X_tr, y_tr, X_va, y_va, X_te, y_te


def run_sanity_checks_all_horizons(bundle_60: Dict[str, Any], bundle_90: Dict[str, Any]) -> None:
    """Corre sanity checks para ambos horizontes."""
    run_sanity_checks_for_bundle(bundle_60, tag="h60")
    run_sanity_checks_for_bundle(bundle_90, tag="h90")

In [96]:
run_sanity_checks_all_horizons(bundle_60, bundle_90)

OK h60 (h=60) | seq_len=29 | n_features=7
OK h90 (h=90) | seq_len=29 | n_features=7


## **8. Definición del modelo — placeholder**

Variante A: baseline naive (ejemplo listo)

In [97]:
def predict_naive_last_value(X: np.ndarray, *, seq_len: int) -> np.ndarray:
    """
    Baseline naive:
    - Devuelve una secuencia de predicción constante.
    - Por ahora: baseline = 0 en todos los pasos de la secuencia.
      (Usted puede cambiarlo luego por 'last close', etc.)
    """
    # Número de muestras (ventanas)
    n = int(X.shape[0])

    # Predicción: todo ceros con shape (n, seq_len)
    y_pred = np.zeros((n, seq_len), dtype=float)

    # Retorna predicciones
    return y_pred

In [98]:
from typing import Any, Dict
import numpy as np

def run_naive_for_bundle(bundle: Dict[str, Any]) -> Dict[str, np.ndarray]:
    """
    Corre naive SOLO para VALID (coherente con el esquema Train/Valid del libro).

    Retorna:
    {
      "y_pred_valid": ...,
    }
    """
    X_valid = bundle["valid"]["X"]
    seq_len = int(X_valid.shape[1])

    y_pred_valid = predict_naive_last_value(X_valid, seq_len=seq_len)

    return {"y_pred_valid": y_pred_valid}

In [99]:
preds_60 = run_naive_for_bundle(bundle_60)
print("H60 preds valid:", preds_60["y_pred_valid"].shape)


H60 preds valid: (195, 29)


In [100]:
preds_90 = run_naive_for_bundle(bundle_90)
print("H90 preds valid:", preds_90["y_pred_valid"].shape)

H90 preds valid: (195, 29)


## **9. Métricas ML**

In [101]:
import pandas as pd

def metrics_to_df(metrics: dict, *, model: str, split: str, horizon: int) -> pd.DataFrame:
    return pd.DataFrame([{
        "model": model,
        "split": split,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA_last": metrics.get("DA_last"),
    }])

# -------- H60 (VALID ONLY) --------
Y_valid_60 = bundle_60["valid"]["y"]
preds_60 = run_naive_for_bundle(bundle_60)
y_pred_valid_60 = preds_60["y_pred_valid"]

ml_valid_60 = compute_seq2seq_metrics(Y_valid_60, y_pred_valid_60, compute_r2=True)

# -------- H90 (VALID ONLY) --------
Y_valid_90 = bundle_90["valid"]["y"]
preds_90 = run_naive_for_bundle(bundle_90)
y_pred_valid_90 = preds_90["y_pred_valid"]

ml_valid_90 = compute_seq2seq_metrics(Y_valid_90, y_pred_valid_90, compute_r2=True)

# -------- Tabla final (solo valid) --------
df_valid_60 = metrics_to_df(ml_valid_60, model="naive", split="valid", horizon=60)
df_valid_90 = metrics_to_df(ml_valid_90, model="naive", split="valid", horizon=90)

pd.concat([df_valid_60, df_valid_90], ignore_index=True)


,model,split,horizon_min,MAE,RMSE,R2,DA_last
0,naive,valid,60,36.446640,50.134159,-0.000116,NaN
1,naive,valid,90,53.419894,70.911865,-0.000004,NaN


In [102]:
# Targets reales desde el bundle
Y_valid_60 = bundle_60["valid"]["y"]

# Predicciones ya calculadas
y_pred_valid_60 = preds_60["y_pred_valid"]

# Métricas seq2seq
ml_valid_60 = compute_seq2seq_metrics(Y_valid_60, y_pred_valid_60, compute_r2=True)

In [103]:
# Targets reales desde el bundle
Y_valid_90 = bundle_90["valid"]["y"]

# Predicciones ya calculadas
y_pred_valid_90 = preds_90["y_pred_valid"]

# Métricas seq2seq
ml_valid_90 = compute_seq2seq_metrics(Y_valid_90, y_pred_valid_90, compute_r2=True)


In [104]:
df_valid_60 = metrics_to_df(ml_valid_60, model="naive", split="valid", horizon=60)
df_valid_90 = metrics_to_df(ml_valid_90, model="naive", split="valid", horizon=90)

pd.concat([df_valid_60, df_valid_90], ignore_index=True)

,model,split,horizon_min,MAE,RMSE,R2,DA_last
0,naive,valid,60,36.446640,50.134159,-0.000116,NaN
1,naive,valid,90,53.419894,70.911865,-0.000004,NaN


## **11. Guardar artefactos para Stage_08**

In [105]:
def save_json(obj: Dict[str, Any], path: Path) -> None:
    """Guarda un diccionario como JSON, creando directorios si es necesario."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

In [106]:
from pathlib import Path
from typing import Any, Dict
import numpy as np

def save_evaluation_artifacts(
    *,
    out_dir: Path,
    model_name: str,
    horizon: int,
    ml_valid: Dict[str, Any],
    y_valid: np.ndarray | None = None,
    y_pred_valid: np.ndarray | None = None,
    save_preds: bool = True,
) -> None:
    """
    Guarda SOLO artefactos de VALID (coherente con el esquema del libro):
      - metrics_ml_valid.json
      - pred_valid.npz (opcional)

    No guarda nada de TEST.
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # Métricas ML (VALID)
    # -------------------------
    save_json(
        {
            "model": model_name,
            "horizon_min": horizon,
            "split": "valid",
            **ml_valid,
        },
        out_dir / "metrics_ml_valid.json",
    )

    # -------------------------
    # Predicciones OOS (VALID) - opcional
    # -------------------------
    if save_preds:
        if y_valid is None or y_pred_valid is None:
            raise ValueError("Si save_preds=True, debe pasar y_valid y y_pred_valid.")
        np.savez_compressed(
            out_dir / "pred_valid.npz",
            y_true=np.asarray(y_valid),
            y_pred=np.asarray(y_pred_valid),
        )

    print(f"OK - artefactos guardados en: {out_dir}")

In [107]:
OUT_DIR_60 = Path("artifacts/01_naive/h60")
OUT_DIR_60 = DRIVE_DIR / OUT_DIR_60 #Solo para la notebook

save_evaluation_artifacts(
    out_dir=OUT_DIR_60,
    model_name="naive",
    horizon=60,
    ml_valid=ml_valid_60,
    y_valid=Y_valid_60,
    y_pred_valid=y_pred_valid_60,
    save_preds=False,  #True si quieres guardar predicciones
)

OK - artefactos guardados en: /content/drive/MyDrive/neural_profit/artifacts/01_naive/h60


In [108]:
OUT_DIR_90 = Path("artifacts/01_naive/h90")
OUT_DIR_90 = DRIVE_DIR / OUT_DIR_90 #Solo para la notebook

save_evaluation_artifacts(
    out_dir=OUT_DIR_90,
    model_name="naive",
    horizon=90,
    ml_valid=ml_valid_90,
    y_valid=Y_valid_90,
    y_pred_valid=y_pred_valid_90,
    save_preds=False,  #True si quieres guardar predicciones
)

OK - artefactos guardados en: /content/drive/MyDrive/neural_profit/artifacts/01_naive/h90


## **12. Resumen**

- **Métricas ML (valid):** MAE, RMSE, DA_last, R2
- Artefactos guardados en `artifacts/<model_name>/h{H}`